In [81]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pickle
import os
import uuid

tf.autograph.set_verbosity(0)  # Tắt autograph tracing

In [82]:
# Check PyTorch version and GPU availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch version: 2.7.0+cu126
CUDA available: True
GPU Name: NVIDIA GeForce GTX 1650


In [83]:
# Đọc dữ liệu
df = pd.read_csv('dataset_LOSO/train.csv')

# Tách tập Test với SUB_ID = 9
test_df = df[df['SUB_ID'] == 9]
train_val_df = df[df['SUB_ID'] != 9]

In [84]:
# Define Attention Layer
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention_weight = nn.Parameter(torch.randn(hidden_dim, 1))
        self.attention_bias = nn.Parameter(torch.zeros(1))
    
    def forward(self, x):
        # x: (batch, seq_len, hidden_dim)
        e = torch.tanh(torch.matmul(x, self.attention_weight) + self.attention_bias)  # (batch, seq_len, 1)
        a = torch.softmax(e, dim=1)  # (batch, seq_len, 1)
        output = x * a  # (batch, seq_len, hidden_dim)
        return torch.sum(output, dim=1)  # (batch, hidden_dim)

# Define Bi-GRU + Attention Model
class BiGRUAttentionModel(nn.Module):
    def __init__(self, input_dim, num_classes, gru_units=64, dropout_rate=0.5):
        super(BiGRUAttentionModel, self).__init__()
        self.gru = nn.GRU(input_dim, gru_units, bidirectional=True, batch_first=True)
        self.attention = Attention(gru_units * 2)  # *2 for bidirectional
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(gru_units * 2, num_classes)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        gru_out, _ = self.gru(x)  # (batch, seq_len, gru_units * 2)
        attn_out = self.attention(gru_out)  # (batch, gru_units * 2)
        out = self.dropout(attn_out)
        out = self.fc(out)  # (batch, num_classes)
        out = self.softmax(out)
        return out

# Create sequences (5 frames per sample)
def create_sequences(X, y, seq_length=5):
    X_seq = []
    y_seq = []
    for i in range(len(X) - seq_length + 1):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length-1])  # Label of the last frame
    return np.array(X_seq), np.array(y_seq)

In [85]:
# Grid Search function with GroupKFold
def grid_search(train_val_df, groups, input_shape, num_classes, device):
    param_grid = {
        'gru_units': [32, 64, 128],
        'learning_rate': [0.001, 0.0005, 0.0001],
        'dropout_rate': [0.3, 0.5, 0.7]
    }

    best_params = None
    best_val_accuracy = 0
    some_threshold = 0.4  # Threshold for val_loss

    # Initialize GroupKFold
    group_kfold = GroupKFold(n_splits=len(np.unique(groups)))

    for params in ParameterGrid(param_grid):
        fold_val_accuracies = []
        fold_val_losses = []

        for fold, (train_idx, val_idx) in enumerate(group_kfold.split(train_val_df.drop(['SUB_ID', 'label'], axis=1), train_val_df['label'], groups)):
            val_subject = np.unique(groups[val_idx])[0]
            print(f"\n=== Grid Search - Fold {fold + 1} (Validation SUB_ID: {val_subject}) ===")

            # Split data
            train_df = train_val_df.iloc[train_idx]
            val_df = train_val_df.iloc[val_idx]

            X_train = train_df.drop(['SUB_ID', 'label'], axis=1).values
            y_train = train_df['label'].values
            X_val = val_df.drop(['SUB_ID', 'label'], axis=1).values
            y_val = val_df['label'].values

            # Standardize
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            # Create sequences
            seq_length = 5
            X_train_seq, y_train_seq = create_sequences(X_train, y_train, seq_length)
            X_val_seq, y_val_seq = create_sequences(X_val, y_val, seq_length)

            # One-hot encode labels
            encoder = OneHotEncoder(sparse_output=False)
            y_train_seq = encoder.fit_transform(y_train_seq.reshape(-1, 1))
            y_val_seq = encoder.transform(y_val_seq.reshape(-1, 1))

            # Compute class weights
            y_train_labels = np.argmax(y_train_seq, axis=1)
            class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_labels), y=y_train_labels)

            # Convert to PyTorch tensors
            X_train_tensor = torch.FloatTensor(X_train_seq).to(device)
            y_train_tensor = torch.LongTensor(np.argmax(y_train_seq, axis=1)).to(device)
            X_val_tensor = torch.FloatTensor(X_val_seq).to(device)
            y_val_tensor = torch.LongTensor(np.argmax(y_val_seq, axis=1)).to(device)

            train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
            val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
            train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=32)

            # Build model
            model = BiGRUAttentionModel(
                input_dim=input_shape[-1],
                num_classes=num_classes,
                gru_units=params['gru_units'],
                dropout_rate=params['dropout_rate']
            ).to(device)

            optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'])
            criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))

            # Early stopping
            patience = 5
            best_val_loss = float('inf')
            counter = 0
            best_model_state = None

            for epoch in range(1000):
                model.train()
                for batch_X, batch_y in train_loader:
                    optimizer.zero_grad()
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                    loss.backward()
                    optimizer.step()

                # Validation
                model.eval()
                val_loss = 0
                correct = 0
                total = 0
                with torch.no_grad():
                    for batch_X, batch_y in val_loader:
                        outputs = model(batch_X)
                        val_loss += criterion(outputs, batch_y).item()
                        _, predicted = torch.max(outputs, 1)
                        total += batch_y.size(0)
                        correct += (predicted == batch_y).sum().item()
                
                val_loss /= len(val_loader)
                val_accuracy = correct / total

                # Early stopping check
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_model_state = model.state_dict()
                    counter = 0
                else:
                    counter += 1
                    if counter >= patience:
                        break

            fold_val_accuracies.append(val_accuracy)
            fold_val_losses.append(best_val_loss)

        # Average metrics across folds
        avg_val_accuracy = np.mean(fold_val_accuracies)
        avg_val_loss = np.mean(fold_val_losses)

        if avg_val_accuracy > best_val_accuracy and avg_val_loss < some_threshold:
            best_val_accuracy = avg_val_accuracy
            best_params = params

        print(f"Parameters: {params}, Avg validation accuracy: {avg_val_accuracy:.4f}, Avg validation loss: {avg_val_loss:.4f}")

    print(f"\nBest parameters: {best_params}, Best average validation accuracy: {best_val_accuracy:.4f}, Best average validation loss: {avg_val_loss:.4f}")
    return best_params

In [86]:
# Main processing
subjects = train_val_df['SUB_ID'].unique()
num_classes = len(df['label'].unique())
print(f"Number of classes: {num_classes}")
print(f"Number of subjects: {len(subjects)}")

Number of classes: 6
Number of subjects: 9


In [87]:
# Grid search with GroupKFold
groups = train_val_df['SUB_ID'].values
best_params = grid_search(
    train_val_df=train_val_df,
    groups=groups,
    input_shape=(5, train_val_df.drop(['SUB_ID', 'label'], axis=1).shape[1]),
    num_classes=num_classes,
    device=device
)

print("\nFinal best parameters after GroupKFold grid search:", best_params)


=== Grid Search - Fold 1 (Validation SUB_ID: 4) ===

=== Grid Search - Fold 2 (Validation SUB_ID: 0) ===

=== Grid Search - Fold 3 (Validation SUB_ID: 6) ===

=== Grid Search - Fold 4 (Validation SUB_ID: 3) ===

=== Grid Search - Fold 5 (Validation SUB_ID: 5) ===

=== Grid Search - Fold 6 (Validation SUB_ID: 1) ===

=== Grid Search - Fold 7 (Validation SUB_ID: 2) ===

=== Grid Search - Fold 8 (Validation SUB_ID: 7) ===

=== Grid Search - Fold 9 (Validation SUB_ID: 8) ===
Parameters: {'dropout_rate': 0.3, 'gru_units': 32, 'learning_rate': 0.001}, Avg validation accuracy: 0.8805, Avg validation loss: 1.1350

=== Grid Search - Fold 1 (Validation SUB_ID: 4) ===

=== Grid Search - Fold 2 (Validation SUB_ID: 0) ===

=== Grid Search - Fold 3 (Validation SUB_ID: 6) ===

=== Grid Search - Fold 4 (Validation SUB_ID: 3) ===

=== Grid Search - Fold 5 (Validation SUB_ID: 5) ===

=== Grid Search - Fold 6 (Validation SUB_ID: 1) ===

=== Grid Search - Fold 7 (Validation SUB_ID: 2) ===

=== Grid Searc

In [88]:
# Prepare full training and test data
X_train_full = train_val_df.drop(['SUB_ID', 'label'], axis=1).values
y_train_full = train_val_df['label'].values
X_test = test_df.drop(['SUB_ID', 'label'], axis=1).values
y_test = test_df['label'].values

# Standardize
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test = scaler.transform(X_test)

# Create sequences for full training and test sets
X_train_full_seq, y_train_full_seq = create_sequences(X_train_full, y_train_full, seq_length=5)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, seq_length=5)

# One-hot encode labels
encoder = OneHotEncoder(sparse_output=False)
y_train_full_seq = encoder.fit_transform(y_train_full_seq.reshape(-1, 1))
y_test_seq = encoder.transform(y_test_seq.reshape(-1, 1))

# Compute class weights for full training
y_train_full_labels = np.argmax(y_train_full_seq, axis=1)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_full_labels), y=y_train_full_labels)

In [91]:
# Group K-Fold Cross-Validation (each fold is one subject)
group_kfold = GroupKFold(n_splits=len(subjects))
best_model = None
best_val_loss = float('inf')
best_model_state = None
best_train_metrics = None  # To store training metrics of the best model

# Create groups for GroupKFold based on SUB_ID
groups = train_val_df['SUB_ID'].values
sequence_groups = []
for i in range(len(X_train_full) - seq_length + 1):
    sequence_groups.append(groups[i + seq_length - 1])  # Use SUB_ID of the last frame
sequence_groups = np.array(sequence_groups)

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train_full_seq, y_train_full_seq, groups=sequence_groups)):
    if best_params is None:
        print("best_params is None, skipping this fold...")
        continue

    # Get the SUB_ID for the validation fold
    val_subject = np.unique(sequence_groups[val_idx])[0]
    print(f"\n=== Fold {fold + 1} (Validation SUB_ID: {val_subject}) ===")

    X_train_cv = X_train_full_seq[train_idx]
    y_train_cv = y_train_full_seq[train_idx]
    X_val_cv = X_train_full_seq[val_idx]
    y_val_cv = y_train_full_seq[val_idx]

    # Convert to tensors
    X_train_cv_tensor = torch.FloatTensor(X_train_cv).to(device)
    y_train_cv_tensor = torch.LongTensor(np.argmax(y_train_cv, axis=1)).to(device)
    X_val_cv_tensor = torch.FloatTensor(X_val_cv).to(device)
    y_val_cv_tensor = torch.LongTensor(np.argmax(y_val_cv, axis=1)).to(device)

    train_dataset = TensorDataset(X_train_cv_tensor, y_train_cv_tensor)
    val_dataset = TensorDataset(X_val_cv_tensor, y_val_cv_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)

    # Build model
    model = BiGRUAttentionModel(
        input_dim=X_train_full_seq.shape[2],
        num_classes=num_classes,
        gru_units=best_params['gru_units'],
        dropout_rate=best_params['dropout_rate']
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=best_params['learning_rate'])
    # Group K-Fold Cross-Validation (each fold is one subject)
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))

group_kfold = GroupKFold(n_splits=len(subjects))
best_model = None
best_val_loss = float('inf')
best_model_state = None
best_train_metrics = None  # To store training metrics of the best model

# Create groups for GroupKFold based on SUB_ID
groups = train_val_df['SUB_ID'].values
sequence_groups = []
for i in range(len(X_train_full) - seq_length + 1):
    sequence_groups.append(groups[i + seq_length - 1])  # Use SUB_ID of the last frame
sequence_groups = np.array(sequence_groups)

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train_full_seq, y_train_full_seq, groups=sequence_groups)):
    if best_params is None:
        print("best_params is None, skipping this fold...")
        continue

    # Get the SUB_ID for the validation fold
    val_subject = np.unique(sequence_groups[val_idx])[0]
    print(f"\n=== Fold {fold + 1} (Validation SUB_ID: {val_subject}) ===")

    X_train_cv = X_train_full_seq[train_idx]
    y_train_cv = y_train_full_seq[train_idx]
    X_val_cv = X_train_full_seq[val_idx]
    y_val_cv = y_train_full_seq[val_idx]

    # Convert to tensors
    X_train_cv_tensor = torch.FloatTensor(X_train_cv).to(device)
    y_train_cv_tensor = torch.LongTensor(np.argmax(y_train_cv, axis=1)).to(device)
    X_val_cv_tensor = torch.FloatTensor(X_val_cv).to(device)
    y_val_cv_tensor = torch.LongTensor(np.argmax(y_val_cv, axis=1)).to(device)

    train_dataset = TensorDataset(X_train_cv_tensor, y_train_cv_tensor)
    val_dataset = TensorDataset(X_val_cv_tensor, y_val_cv_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32)

    # Build model
    model = BiGRUAttentionModel(
        input_dim=X_train_full_seq.shape[2],
        num_classes=num_classes,
        gru_units=best_params['gru_units'],
        dropout_rate=best_params['dropout_rate']
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=best_params['learning_rate'])
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))

    # Early stopping
    patience = 10
    counter = 0
    fold_best_val_loss = float('inf')
    fold_best_train_loss = float('inf')
    fold_best_train_accuracy = 0
    fold_best_model_state = None

    for epoch in range(50):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += batch_y.size(0)
            train_correct += (predicted == batch_y).sum().item()

        train_loss /= len(train_loader)
        train_accuracy = train_correct / train_total

        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                outputs = model(batch_X)
                val_loss += criterion(outputs, batch_y).item()
                _, predicted = torch.max(outputs, 1)
                val_total += batch_y.size(0)
                val_correct += (predicted == batch_y).sum().item()
        
        val_loss /= len(val_loader)
        val_accuracy = val_correct / val_total

        # Print metrics for each epoch
        print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

        # Update best model for this fold
        if val_loss < fold_best_val_loss:
            fold_best_val_loss = val_loss
            fold_best_train_loss = train_loss
            fold_best_train_accuracy = train_accuracy
            fold_best_model_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    # Update global best model
    if fold_best_val_loss < best_val_loss:
        best_val_loss = fold_best_val_loss
        best_model = model
        best_model_state = fold_best_model_state
        best_train_metrics = {
            'train_loss': fold_best_train_loss,
            'train_accuracy': fold_best_train_accuracy
        }


best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...
best_params is None, skipping this fold...


In [92]:
# Print best model metrics
if best_model is not None:
    print(f"\n=== Best Model Metrics ===")
    print(f"Best Validation Loss: {best_val_loss:.4f}")
    print(f"Best Training Loss: {best_train_metrics['train_loss']:.4f}")
    print(f"Best Training Accuracy: {best_train_metrics['train_accuracy']:.4f}")

    # Save the best model
    torch.save(best_model_state, "Model/Squat_detection_GRU_LOSO_v1.pt")
else:
    print("No best model found. Please check the training process.")

No best model found. Please check the training process.


In [93]:
# Evaluate on test set
if best_model is not None:
    best_model.load_state_dict(best_model_state)
    best_model.eval()
    X_test_tensor = torch.FloatTensor(X_test_seq).to(device)
    y_test_tensor = torch.LongTensor(np.argmax(y_test_seq, axis=1)).to(device)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    test_loader = DataLoader(test_dataset, batch_size=32)

    test_loss = 0
    correct = 0
    total = 0
    y_pred = []
    y_true = []
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(device))

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = best_model(batch_X)
            test_loss += criterion(outputs, batch_y).item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            y_pred.extend(predicted.cpu().numpy())
            y_true.extend(batch_y.cpu().numpy())

    test_loss /= len(test_loader)
    test_accuracy = correct / total

    # Classification report
    report = classification_report(y_true, y_pred, output_dict=True)
    print("\n=== Final Evaluation for Test (SUB_ID=9) ===")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print("\nClassification Report:")
    print(pd.DataFrame(report).T)
else:
    print("No best model found for evaluation.")

No best model found for evaluation.


In [94]:
# Save scaler
scaler_file = 'Model/scaler_GRU_LOSO_v1.pkl'
with open(scaler_file, 'wb') as f:
    pickle.dump(scaler, f)

# Verify scaler loading
try:
    with open(scaler_file, 'rb') as f:
        scaler = pickle.load(f)
    print("Scaler loaded successfully!")
except Exception as e:
    print(f"Error loading scaler: {e}")

Scaler loaded successfully!
